# 🎬 Test đếm trên VIDEO THẬT — có LƯU VIDEO OUTPUT

Chạy detect→track→đếm trên video công khai + **lưu video output** (vẽ vạch/vùng +
box + track-id + số đếm) để bạn kiểm tra mắt thường. Người (market-square, subway…)
tách **2 bài: cắt VẠCH (vào/ra)** và **đếm trong VÙNG (occupancy)**.


## 1) Tải code + cài thư viện


In [ ]:
%cd /kaggle/working
!rm -rf VisionOS
!git clone -q https://github.com/nguyendinhhuyht20032004-ai/VisionOS.git
%cd VisionOS/VisionOS
!git checkout -q claude/rebuild-visionos-codebase-tgg0mf
!git pull -q origin claude/rebuild-visionos-codebase-tgg0mf
!pip install -q ultralytics 'supervision>=0.21' opencv-python-headless


## 2) Xem catalog (vạch/vùng + query của mỗi video)


In [ ]:
!python run_scenarios.py --list


## 3) 🚗 Đếm XE + lưu video output
Video lưu vào `/kaggle/working/scen_out/vehicles/`.


In [ ]:
!python run_scenarios.py --task vehicles --max-frames 300 --save-dir /kaggle/working/scen_out


## 4) 🚶 Đếm NGƯỜI — cắt VẠCH (vào/ra) + đếm VÙNG
market-square chạy CẢ 2 bài (vạch & vùng); subway/siêu thị/đi-bộ đếm vào/ra.
Xem video output để chỉnh vạch/vùng nếu đếm lệch.


In [ ]:
!python run_scenarios.py --task people --max-frames 300 --save-dir /kaggle/working/scen_out


## 5) 📦 Đếm SẢN PHẨM — chai trên chuyền (YOLO, nhanh) + lưu video


In [ ]:
!python run_scenarios.py --task conveyor --only milk --max-frames 300 --save-dir /kaggle/working/scen_out


## 6) 📦🧠 Đếm SẢN PHẨM với QUERY KHÓ (open-vocab) + lưu video
Chạy chậm (LocateAnything + auto-pin transformers). Bỏ `#` để chạy.


In [ ]:
# !python run_scenarios.py --task conveyor --all-queries --max-frames 80 --save-dir /kaggle/working/scen_out


## 7) 🎥 XEM / TẢI video output
Liệt kê video đã lưu; chuyển 1 video sang H.264 để xem ngay trong notebook.
Tải cả thư mục: panel phải Kaggle → Output → `scen_out`.


In [ ]:
import glob, os
from IPython.display import Video, display
vids = sorted(glob.glob('/kaggle/working/scen_out/**/*.mp4', recursive=True))
print(f'{len(vids)} video output:')
for p in vids: print('  ', p)
# chuyển 1 video sang H.264 để phát inline (Kaggle không phát mp4v):
if vids:
    src = vids[0]; dst = '/kaggle/working/preview_h264.mp4'
    os.system(f'ffmpeg -y -loglevel error -i "{src}" -vcodec libx264 -pix_fmt yuv420p "{dst}"')
    print('Xem:', src); display(Video(dst, embed=True, width=700))


---
### Đọc kết quả
- Scorecard tách **cắt VẠCH** (IN/OUT/total) và **VÙNG** (trong_vùng/đỉnh).
- Video output: **vàng=vạch**, **xanh mờ=vùng**, **xanh dương=box + #track-id**, banner số đếm.
- Đếm lệch? Xem video → chỉnh `line_start_pct/line_end_pct` hoặc `zone_points_pct` trong `recognition/video_catalog.py`.
